# Шаг 18. Построение агрегированных таблиц

Агрегации будут применениы для ответа на следующие аналитические вопросы из Шага 2:

**1. Какие производители выпускают наибольшее количество наборов? Какие 20% производителей дают 80% всего ассортимента?**

**agg_manufacturers (ABC-анализ)** позволяет отсечь "длинный хвост" (категория C) и сфокусироваться на производителях категории A, которые формируют 80% ассортимента.

**2. Какие исторические эпохи наиболее представлены на рынке? На какие периоды приходится основная доля всех выпущенных наборов?**

**3. Какие национальности чаще всего встречаются в наборах? Есть ли национальности, представленные в менее чем 5 наборах (нишевые интересы)?**

**agg_era_nat (Карта эпох и национальностей)** - подготовка данных для Экрана 1 дашборда.

**4. Как менялось количество выпускаемых наборов по годам? Есть ли периоды роста или спада активности производителей?**

**agg_decades (Динамика)** показывает пики и спады в индустрии, а также тренд изменения среднего качества (avg_rating) со временем.

**10. Есть ли связь между количеством фигур в наборе и рейтингом качества? Большие наборы (50+ фигур) оцениваются выше или ниже малых (12–24 фигуры)?**

**agg_figures (Связь размера и качества)** позволяет проверить оцениваются ли большие наборы (50+ фигур) выше, чем маленькие, или качество не зависит от размера.

**13. FM-анализ активности производителей: Для каждой исторической темы (национальность или период) — как давно производитель выпускал последний набор (Recency), сколько всего наборов по теме он выпустил (Frequency), и каков средний рейтинг качества (Magnitude вместо Monetary)?**

**agg_rfm (Адаптированный RFM)** - ключевая таблица для навигации по коллекционированию.

In [3]:
import pandas as pd
import numpy as np

# 1. Загружаем очищенные данные с правильными типами
df_sets = pd.read_csv('df_sets.csv', dtype={
    'release_year': 'Int64',
    'aggregate_rating': 'Int64',
    'num_figures': 'Int64',
    'decade': 'Int64',
    'years_since_release': 'Int64'
})

df_long = pd.read_csv('df_consolidated_clean.csv', dtype={
    'release_year': 'Int64',
    'aggregate_rating': 'Int64',
    'num_figures': 'Int64',
    'decade': 'Int64',
    'years_since_release': 'Int64'
})

CURRENT_YEAR = 2026

print("=== 1. ABC-анализ производителей (Структура рынка) ===")
agg_manufacturers = df_sets.groupby('manufacturer').agg(
    kit_count=('id', 'nunique'),
    avg_rating=('aggregate_rating', 'mean'),
    avg_figures=('num_figures', 'mean')
).sort_values('kit_count', ascending=False).reset_index()

total_kits = agg_manufacturers['kit_count'].sum()
agg_manufacturers['cumulative_pct'] = (agg_manufacturers['kit_count'].cumsum() / total_kits * 100).round(1)

def get_abc_category(pct):
    if pct <= 80: return 'A'
    elif pct <= 95: return 'B'
    else: return 'C'

agg_manufacturers['ABC_Category'] = agg_manufacturers['cumulative_pct'].apply(get_abc_category)
print(agg_manufacturers.head(10))


print("\n=== 2. Динамика рынка по десятилетиям ===")
agg_decades = df_sets.groupby('decade').agg(
    kit_count=('id', 'nunique'),
    avg_rating=('aggregate_rating', 'mean')
).reset_index().sort_values('decade')

agg_decades = agg_decades[agg_decades['decade'].notna()]
print(agg_decades)


print("\n=== 3. Адаптированный RFM-анализ: Актуальные хиты (Производитель + Национальность) ===")
# Фильтр: убираем "Не указано", оставляем только наборы с годом и рейтингом
df_rfm = df_long[
    (df_long['release_year'].notna()) & 
    (df_long['aggregate_rating'].notna()) & 
    (df_long['nationality'] != 'Не указано')
].copy()

agg_rfm = df_rfm.groupby(['manufacturer', 'nationality']).agg(
    # R (Recency): сколько лет прошло с последнего релиза
    Recency=('release_year', lambda x: CURRENT_YEAR - x.max()),
    # F (Frequency): сколько всего уникальных наборов выпущено
    Frequency=('id', 'nunique'),
    # M (Magnitude): средний рейтинг качества
    Magnitude=('aggregate_rating', 'mean')
).reset_index()

# ИЗМЕНЕНИЕ №3: показываем "Актуальные хиты"
# Сортировка: сначала самые свежие релизы (Recency ASC), затем по убыванию рейтинга (Magnitude DESC)
agg_rfm_hits = agg_rfm.sort_values(['Recency', 'Magnitude'], ascending=[True, False])

# Дополнительно: отбираем только действительно актуальные хиты
# Recency <= 5 лет (выпущены в 2021-2026) и Magnitude >= 40 (высокий рейтинг)
agg_rfm_hits = agg_rfm_hits[
    (agg_rfm_hits['Recency'] <= 5) & 
    (agg_rfm_hits['Magnitude'] >= 40)
]

print(f"Найдено актуальных хитов (Recency ≤ 5 лет, Magnitude ≥ 40): {len(agg_rfm_hits)}")
print(agg_rfm_hits.head(20))


print("\n=== 4. Карта исторических периодов и национальностей (для Экрана 1 дашборда) ===")
# ИЗМЕНЕНИЕ №2: переименование и хронологический порядок
df_era_nat = df_long[df_long['nationality'] != 'Не указано'].copy()

# Задаём хронологический порядок эр через категориальный тип
era_order = ['Древний мир', 'Средневековье', 'Новое время', 'Новейшее время', 'Современность']
df_era_nat['era'] = pd.Categorical(df_era_nat['era'], categories=era_order, ordered=True)

agg_era_nat = df_era_nat.groupby(['era', 'nationality']).agg(
    kit_count=('id', 'nunique')
).reset_index()

# Сортировка: сначала по хронологии эр, затем по убыванию количества наборов
agg_era_nat = agg_era_nat.sort_values(['era', 'kit_count'], ascending=[True, False])

# Показываем топ-5 национальностей в каждом периоде
top_nat_per_era = agg_era_nat.groupby('era').head(5)
print(top_nat_per_era)


print("\n=== 5. Связь количества фигур и рейтинга (Качество наборов) ===")
df_sets['figure_category'] = pd.cut(
    df_sets['num_figures'], 
    bins=[0, 15, 30, 50, 9999], 
    labels=['Мини (1-15)', 'Стандарт (16-30)', 'Крупный (31-50)', 'Сэмплер/Диорама (51+)']
)

# ИЗМЕНЕНИЕ №1: добавлен observed=True для подавления FutureWarning
agg_figures = df_sets.groupby('figure_category', observed=True).agg(
    kit_count=('id', 'nunique'),
    avg_rating=('aggregate_rating', 'mean'),
    median_figures=('num_figures', 'median')
).reset_index()
print(agg_figures)


print("\n=== 6. Сохранение агрегированных таблиц ===")
agg_manufacturers.to_csv('agg_manufacturers_abc.csv', index=False, encoding='utf-8')
agg_decades.to_csv('agg_decades_dynamics.csv', index=False, encoding='utf-8')
agg_rfm_hits.to_csv('agg_rfm_actual_hits.csv', index=False, encoding='utf-8')  # переименован файл
agg_era_nat.to_csv('agg_era_nationality_map.csv', index=False, encoding='utf-8')
agg_figures.to_csv('agg_figure_categories.csv', index=False, encoding='utf-8')

print("✅ Все 5 агрегированных таблиц успешно сохранены в формате CSV.")

=== 1. ABC-анализ производителей (Структура рынка) ===
  manufacturer  kit_count  avg_rating  avg_figures  cumulative_pct  \
0     Strelets        448   41.849188    37.002262            15.8   
1          HaT        348   41.454545    35.565625            28.0   
2       Zvezda        176   45.923077    14.841176            34.2   
3       RedBox        155   39.945946    30.503268            39.7   
4         Mars        145   35.446429    33.239669            44.8   
5      Italeri        120    42.40625    32.295918            49.0   
6       Caesar        118       45.71    31.034483            53.2   
7      Preiser         94   46.766667    15.011111            56.5   
8       Revell         84   44.882353    39.318841            59.4   
9     Linear-A         79   42.859155     24.74359            62.2   

  ABC_Category  
0            A  
1            A  
2            A  
3            A  
4            A  
5            A  
6            A  
7            A  
8            A  
9   

C:\Users\mi\AppData\Local\Temp\ipykernel_12152\2982727122.py:92: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_era_nat = df_era_nat.groupby(['era', 'nationality']).agg(
C:\Users\mi\AppData\Local\Temp\ipykernel_12152\2982727122.py:100: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  top_nat_per_era = agg_era_nat.groupby('era').head(5)


## Результат шага
Подготовлены 5 чистых, структурированных CSV-файлов с агрегированными показателями. Эти файлы являются источником данных (Data Source) для подключения к BI-системе, так как они уже сгруппированы, очищены и содержат рассчитанные метрики (накопительный процент, RFM-метрики, категории).